# 第 2 章 采样函数与离散傅里叶变换 · 代码实战（Python）

> 本书章代码的 Python 实现，逐段中文注释；对应 MATLAB 版本见 `matlab/fourier/ch2/`。

## DFT · 定标

`ch2_dft_scaling.py`

In [ ]:
# ===== 书 2.3 / 2.5 节：DFT 定标与周期延拓（式 2.19-2.26，复现图 2.6） =====
# 书图 2.6 参数：L = 20, Δx = 1, N = 20 个样本，f(x) = rect(x/5)（半宽 2.5）。
# 验证三件事：
#   (1) 式 (2.19) 的 DFT 定义（含 exp[-j2π(pm/M+qn/N)] 核）与 np.fft.fft 逐元素一致
#       （前提：数据先排成"移位排列"——index 0 对应坐标 0，见 2.4 节）；
#   (2) 乘上 Δx 后，FFT 样本"跟随"周期延拓谱 F_P（Poisson 求和），
#       而与解析谱 F 的偏差 = 混叠（书 2.5 节图 2.6(b) 的高频瓣略大）；
#   (3) 式 (2.19)+(2.20) 连用（1/MN 因子）恢复原数组（傅里叶积分定理的离散版）。
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "common"))
from base import rect, coords

L = 20.0          # 采样空间边长 [m]（书图 2.6：L=20）
dx = 1.0          # 采样间隔 [m]
N = int(round(L / dx))
x, f = coords(L, N)          # 居中坐标：x ∈ [-10, 9]，f ∈ [-0.5, 0.4]
g = rect(x / 5.0)            # rect 半宽 2.5 → 样本 x=-2..2 共 5 个为 1，面积 5

# ---------- (1) 式 (2.19) 矩阵 DFT vs np.fft.fft（都在移位排列下） ----------
g_shift = np.fft.ifftshift(g)                     # 居中 → 移位（index 0 = 坐标 0）
m = np.arange(N)                                  # 式 (2.19) 的 m = 0..N-1（移位排列）
p = np.arange(N)[:, None]
W = np.exp(-2j * np.pi * p * m / N)               # DFT 核 exp[-j2π pm/M]
G_matrix = W @ g_shift                            # 式 (2.19)（无 Δx 因子）
G_fft = np.fft.fft(g_shift)
err_fft = np.max(np.abs(G_matrix - G_fft))
print(f"[1] 式(2.19)矩阵 DFT 与 np.fft.fft 最大差异 = {err_fft:.2e}")

# ---------- (2) FFT×Δx 样本 = 周期延拓谱 F_P 的样本 ----------
G = np.fft.fftshift(G_fft) * dx                   # 乘 Δx 回连续量纲，再居中排列
F_analytic = 5.0 * np.sinc(5.0 * f)               # 解析谱：rect(x/5) ↔ 5 sinc(5f)
k = np.arange(-3, 4)                              # 周期延拓：F_P(f) = Σ_k F(f - k/Δx)
F_periodic = np.sum([5.0 * np.sinc(5.0 * (f - kk / dx)) for kk in k], axis=0)
err_per = np.max(np.abs(G - F_periodic))
err_alias = np.max(np.abs(G - F_analytic))
print(f"[2] FFT×Δx 与周期谱 F_P 最大差异 = {err_per:.2e}")
print("    （FFT 样本只是'跟随'周期谱：空域矩形窗把 F_s 平滑了一层（式 2.24 的 sinc 卷积），")
print("      加上样本化效应，残差 ~1e-2；书图 2.6(b) 的点与虚线同样只差一点点）")
print(f"    FFT×Δx 与解析谱 F  最大差异 = {err_alias:.3e}（这就是混叠主项，瓣高处偏大）")
print(f"    DC 样本 G(0) = {G[N // 2].real:.6f}（= 中心纵坐标 ∫g dx = 5）")

# ---------- (3) 逆变换恢复原数组（式 2.20 的 1/MN 因子在 ifft 内置） ----------
g_back = np.fft.fftshift(np.fft.ifft(np.fft.ifftshift(G / dx)))
err_inv = np.max(np.abs(g_back - g))
print(f"[3] DFT → DFT^-1 往返恢复原数组最大差异 = {err_inv:.2e}")

# ---------- 画图（复现书图 2.6） ----------
fig, axes = plt.subplots(2, 1, figsize=(8.6, 7.2))
axes[0].plot(x, rect(x / 5.0), "k-", lw=1, label="analytic")
axes[0].plot(x - L, rect((x - L) / 5.0), "k--", lw=0.8, alpha=0.6)
axes[0].plot(x + L, rect((x + L) / 5.0), "k--", lw=0.8, alpha=0.6)
axes[0].plot(x, g, "o", ms=6, color="crimson", mfc="none", label="sampled (N=20)")
axes[0].set_xlim(-30, 30); axes[0].set_ylim(-0.05, 1.3)
axes[0].set_xlabel("x"); axes[0].set_ylabel("f(x)")
axes[0].set_title("Periodic extension: rect, L = 20, dx = 1")
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].plot(f, np.abs(F_analytic), "k-", lw=1.2, label="analytic F")
axes[1].plot(f, np.abs(F_periodic), "k--", lw=0.9, alpha=0.7, label="periodic F_P")
axes[1].plot(f, np.abs(G), "o", ms=6, color="crimson", mfc="none", label="FFT x dx (N=20)")
axes[1].set_xlim(-1.5, 1.5); axes[1].set_ylim(0, 5.6)
axes[1].set_xlabel(r"$f_X$"); axes[1].set_ylabel(r"$|F(f_X)|$")
axes[1].set_title("Spectrum: FFT samples follow the periodic spectrum (aliasing lifts the sidelobes)")
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(os.path.dirname(os.path.abspath(__file__)),
                         "out_periodic_ext.png"), dpi=150)
print("已保存 out_periodic_ext.png")


## 有效 · 带宽

`ch2_eff_bandwidth.py`

In [ ]:
# ===== 书 2.2 节：有效带宽（式 2.9-2.11 + 表 2.1） =====
# 有效带宽定义：包含 98% 总频谱功率的（径向）频谱宽度。
#   P(B)/P_T = 0.98,  P(B) = ∫_0^{2π}∫_0^B |F(ρcosθ, ρsinθ)|^2 ρ dρ dθ   （式 2.10）
# 总功率 P_T 用 Parseval 定理在空域算（式 2.9），数值累积一律除以解析 P_T，
# 避免频域截断把 98% 交点带偏（|F|^2 在 98% 附近非常平坦，0.3% 的归一化偏差
# 就能把 B 推移百分之几十）。
# 书中结果（表 2.1）：rect(x/2w)rect(y/2w) → B≈5/w；circ(sqrt(x^2+y^2)/w) → B≈5/w；
#                     exp[-π(x^2+y^2)/w^2] → B≈0.79/w。
# 圆函数的两种半径约定（承接 ch1 勘误 1）：
#   表 1.2 定义 circ(r/w) 是半径 w/2 的圆盘 → 径向 98% 半径 B·w ≈ 9.97；
#   Goodman/表 1.3 约定（半径 = w 的圆盘）→ B·w ≈ 4.99，与表 2.1 的 5/w 一致。
#   即表 2.1 的 circ 行再次站在"半径 = 分母"的 Goodman 约定一边。
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "common"))
from base import sinc, jinc   # 公共模块：rect/sinc/jinc 等，ch1 起的积木

w = 1.0                        # 特征半宽（结果按 B·w 报告，与 w 无关）
NR = 16001                     # ρ 网格密度
rho = np.linspace(0.0, 60.0 / w, NR)           # 频域半径 [1/m]


def cumtrapz(y, x):
    """0 起头的累积梯形积分（numpy 2.x 无内建 cumtrapz）。"""
    return np.concatenate(([0.0], np.cumsum((y[1:] + y[:-1]) / 2 * np.diff(x))))


# ---------- rect：|F|^2 非圆对称，须做完整 θ 积分（式 2.10） ----------
# 利用正方形对称性：积分 θ∈[0, π/4] 再乘 8（被积函数在 π/4 镜像 + π/2 周期）。
NT = 16384
th = np.linspace(0.0, np.pi / 4, NT)
ring_rect = np.empty(NR)
step = 128
for s in range(0, NR, step):
    r = rho[s:s + step, None]
    fX = r * np.cos(th)[None, :]
    fY = r * np.sin(th)[None, :]
    v = (4.0 * w ** 2 * sinc(2 * w * fX) * sinc(2 * w * fY)) ** 2
    ring_rect[s:s + step] = (v[:, 1:] + v[:, :-1]).sum(axis=1) * (th[1] - th[0]) / 2
ring_rect *= 8.0
P_rect = cumtrapz(ring_rect * rho, rho)        # P(B)，除以解析 P_T=4w² 得占比

# ---------- circ 与 gauss：圆对称，θ 积分 trivially = 2π，直接径向 1D ----------
F2_circ = jinc(rho, w) ** 2                    # 半径 w 圆盘（Goodman/表 1.3 约定）
P_circ = cumtrapz(2 * np.pi * F2_circ * rho, rho)
F2_gauss = (w ** 2 * np.exp(-np.pi * w ** 2 * rho ** 2)) ** 2
P_gauss = cumtrapz(2 * np.pi * F2_gauss * rho, rho)

P_T = {"rect": 4.0 * w ** 2,                   # 式 2.9
       "circ(R=w)": np.pi * w ** 2,            # 圆盘面积
       "circ(R=w/2)": np.pi * (w / 2.0) ** 2,
       "gauss": w ** 2 / 2.0}                  # ∫∫exp(-2πr²/w²)

B_rect = np.interp(0.98 * P_T["rect"], P_rect, rho) * w
B_circ_w = np.interp(0.98 * P_T["circ(R=w)"], P_circ, rho) * w
B_circ_half = B_circ_w * 2                     # 半径减半 → B 加倍（尺度反比）
B_gauss = np.interp(0.98 * P_T["gauss"], P_gauss, rho) * w

# rect 交叉验证：直角网格 + 解析归一化（98% 附近的累积曲线极平坦，
# 极坐标的 θ/ρ 离散误差会放大成 B 的百分之几偏移；直角网格求和更稳）。
half, nc = 40.0, 4001
fc = np.linspace(-half, half, nc)
dfc = fc[1] - fc[0]
acc5 = acc55 = s_tot = 0.0
for s in range(0, nc, 250):
    FYc, FXc = np.meshgrid(fc[s:s + 250], fc, indexing="ij")
    Pc = 16.0 * sinc(2 * FXc) ** 2 * sinc(2 * FYc) ** 2
    Rc = np.hypot(FXc, FYc)
    s_tot += Pc.sum() * dfc ** 2
    acc5 += Pc[Rc <= 5.0].sum() * dfc ** 2
    acc55 += Pc[Rc <= 5.5].sum() * dfc ** 2
fr5, fr55 = acc5 / P_T["rect"], acc55 / P_T["rect"]
B_rect_cart = 5.0 + 0.5 * (0.98 - fr5) / (fr55 - fr5)   # 线性插值出 98% 交点

print("98% 有效带宽（数值，径向判据）")
print(f"  rect(x/2w)rect(y/2w) : B·w = {B_rect:.3f}（极坐标）；"
      f"直角网格参照 {B_rect_cart:.3f}  （书表 2.1：5/w ✓）")
print(f"  circ 半径 w   (Goodman/表 1.3 约定): B·w = {B_circ_w:.3f}（书表 2.1：5/w ✓）")
print(f"  circ 半径 w/2 (表 1.2 定义)        : B·w = {B_circ_half:.3f}")
print(f"  exp(-pi r^2/w^2)     : B·w = {B_gauss:.4f}  （书表 2.1：0.79/w；"
      f"解析 sqrt(ln50/2pi) = {np.sqrt(np.log(50) / (2 * np.pi)):.4f}）")

# circ 用 u=2πRρ 空间交叉验证：P/(πR²) = 2∫_0^{2πRρ} J1²(u)/u du（已知总积分 = 1/2），
# 一维精细梯形即可做到近机器精度（免二维截断误差）。
from scipy.special import j1 as _j1
u = np.linspace(0.0, 400.0, 400001)
vu = np.where(u > 1e-12, _j1(u) ** 2 / np.where(u > 1e-12, u, 1.0), 0.0)
cum_u = np.concatenate(([0.0], np.cumsum((vu[1:] + vu[:-1]) / 2 * np.diff(u))))
u98 = np.interp(0.98 * 0.5, cum_u, u)
print(f"  [交叉验证] u 空间精确解：98% 处 2πRρ = {u98:.2f} → B·R = {u98 / (2 * np.pi):.3f}"
      f"（半径 w 约定 B·w = {u98 / (2 * np.pi):.3f}；半径 w/2 约定 B·w = {u98 / np.pi:.3f}）")

# ---------- 画图：左两幅 = rect 频谱 98% 范围（复现图 2.3），右 = 累积功率曲线 ----------
fmax = 6.0 / w
n = 601
fx1 = np.linspace(-fmax, fmax, n)
FX, FY = np.meshgrid(fx1, fx1)                 # 'xy' 索引，与 MATLAB 一致
Fmag = 4.0 * w ** 2 * sinc(2 * w * FX) * sinc(2 * w * FY)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
im = axes[0].imshow(np.abs(Fmag), extent=[-fmax, fmax, -fmax, fmax], origin="lower",
                    cmap="magma")
axes[0].add_patch(plt.Circle((0, 0), B_rect / w, fill=False, color="cyan", ls="--", lw=1.5))
axes[0].set_title(r"$|F(f_X,f_Y)|$, rect, circle = 98% power")
axes[0].set_xlabel(r"$f_X/w$"); axes[0].set_ylabel(r"$f_Y/w$")
plt.colorbar(im, ax=axes[0], fraction=0.046)

im = axes[1].imshow(np.abs(Fmag) ** 2, extent=[-fmax, fmax, -fmax, fmax], origin="lower",
                    cmap="magma")
axes[1].add_patch(plt.Circle((0, 0), B_rect / w, fill=False, color="cyan", ls="--", lw=1.5))
axes[1].set_title(r"$|F|^2$ (power spectrum), rect")
axes[1].set_xlabel(r"$f_X/w$"); axes[1].set_ylabel(r"$f_Y/w$")
plt.colorbar(im, ax=axes[1], fraction=0.046)

axes[2].plot(rho * w, P_rect / P_T["rect"], label="rect(x/2w)rect(y/2w)")
axes[2].plot(rho * w, P_circ / P_T["circ(R=w)"], label="circ, disk R=w (Table 2.1)")
axes[2].plot(rho * w, P_gauss / P_T["gauss"], label=r"$\exp(-\pi r^2/w^2)$")
axes[2].axhline(0.98, color="k", ls=":", lw=1)
axes[2].plot(B_rect, 0.98, "o", ms=5)
axes[2].plot(B_circ_w, 0.98, "o", ms=5)
axes[2].plot(B_gauss, 0.98, "o", ms=5)
axes[2].annotate(f"{B_rect:.2f}", (B_rect, 0.98), textcoords="offset points", xytext=(-6, 8), fontsize=8)
axes[2].annotate(f"{B_circ_w:.2f}", (B_circ_w, 0.98), textcoords="offset points", xytext=(8, -14), fontsize=8)
axes[2].annotate(f"{B_gauss:.2f}", (B_gauss, 0.98), textcoords="offset points", xytext=(4, -12), fontsize=8)
axes[2].set_xlim(0, 12); axes[2].set_ylim(0.9, 1.001)
axes[2].set_xlabel(r"$B\cdot w$"); axes[2].set_ylabel("cumulative power fraction")
axes[2].set_title("98% effective bandwidth (Table 2.1)")
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(os.path.dirname(os.path.abspath(__file__)),
                         "out_eff_bw.png"), dpi=150)
print("已保存 out_eff_bw.png")


## 网格 · 移位 · 演示

`ch2_grid_shift_demo.py`

In [ ]:
# ===== 书 2.4 节：坐标、索引、居中与移位（式 2.21/2.22，复现图 2.4 / 2.5） =====
# 三种排列（书图 2.4）：
#   analytic ：指标带正负（m = -M/2 .. M/2-1），纸面公式用；
#   centered ：放进计算机后的排列，零坐标在 index M/2+1（MATLAB 1-based）
#              ↔ Python 0-based 的 index N//2；
#   shifted  ：FFT 约定 index 1（MATLAB）/ 0（Python）= 零坐标，FFT 前必须转成它。
# centered ↔ shifted 用 fftshift / ifftshift 互换（偶数 N 两者相同，奇数 N 不同！）。
# 本脚本用 N=8 小网格把三种排列打印出来，并复现图 2.4（1D）与图 2.5（2D 象限交换）。
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "common"))
from base import rect, coords

# ---------- 1D：N=8 小网格，rect 半宽 1.5（3 个样本为 1） ----------
N, L = 8, 8.0
dx = L / N
x, f = coords(L, N)                        # 式 (2.21)/(2.22)：居中升序坐标
g_centered = rect(x / 3.0)                 # 居中排列：|x|<1.5 → x=-1,0,1
g_shifted = np.fft.ifftshift(g_centered)   # 居中 → 移位（FFT 约定）

print("N=8, L=8, dx=1 的三种排列（数值对应书图 2.4）")
print("MATLAB 1-based index:", np.arange(1, N + 1))
print("Python 0-based index:", np.arange(N))
print("coords x (式 2.21)   :", x)
print("g 居中 centered      :", g_centered.astype(int))
print("g 移位 shifted(FFT用):", g_shifted.astype(int))
print("freq 坐标 f (式 2.22):", f)
print("roundtrip fftshift(ifftshift(g)) == g :",
      np.allclose(np.fft.fftshift(g_shifted), g_centered))
print("注意：偶数 N 时 fftshift 与 ifftshift 等价；奇数 N 不等价（见笔记坑与陷阱）")

# ---------- 2D：象限交换（书图 2.5） ----------
N2 = 8
x2, _ = coords(L, N2)
X, Y = np.meshgrid(x2, x2)                # 'xy'：行 ↔ y，列 ↔ x（与 MATLAB 一致）
A_centered = (np.hypot(X, Y) < 1.5).astype(float)   # 居中圆盘，[0,0] 在 (N/2, M/2)
A_shifted = np.fft.ifftshift(A_centered)            # [0,0] 挪到 (0, 0)
print("\n2D 居中排列零坐标位置 (Python 行,列) =",
      tuple(np.argwhere(A_centered > 0).mean(axis=0).round(1)),
      "；MATLAB (i,j) 记法 = (N/2+1, M/2+1) = (5, 5)")
print("2D 移位排列零坐标位置 (Python 行,列) =", tuple(np.argwhere(A_shifted > 0)[0]))

# ---------- 画图：上排复现图 2.4 三种 1D 排列，下排复现图 2.5 象限交换 ----------
fig = plt.figure(figsize=(11.5, 7.6))
gs = fig.add_gridspec(2, 3)

ax = fig.add_subplot(gs[0, 0])
ax.plot(x, g_centered, "o-", ms=5, color="crimson", mfc="none")
ax.set_title("centered: zero coord at index N//2\n(apply ifftshift before fft)")
ax.set_xticks(x); ax.grid(True, alpha=0.3); ax.set_ylim(-0.2, 1.25)

ax = fig.add_subplot(gs[0, 1])
ax.plot(np.arange(N), g_shifted, "s-", ms=5, color="teal", mfc="none")
ax.set_title("shifted for FFT: zero coord at index 0")
ax.set_xticks(np.arange(N)); ax.grid(True, alpha=0.3); ax.set_ylim(-0.2, 1.25)

ax = fig.add_subplot(gs[0, 2])
G = np.fft.fft(g_shifted) * dx
ax.plot(f, np.abs(np.fft.fftshift(G)), "o-", ms=5, color="slateblue", mfc="none")
ax.set_title("|FFT x dx| back on centered freq axis")
ax.grid(True, alpha=0.3)

# 象限编号（书图 2.5：居中 = 3,4/2,1；移位 = 1,2/4,3）
q = np.zeros((N2, N2), int)
q[:N2 // 2, :N2 // 2] = 3; q[:N2 // 2, N2 // 2:] = 4
q[N2 // 2:, :N2 // 2] = 2; q[N2 // 2:, N2 // 2:] = 1
q_shift = np.fft.ifftshift(q)
for k, (A, ttl) in enumerate([(q, "2D centered (quadrant No.)\n[0,0] at MATLAB (N/2+1, M/2+1)"),
                              (q_shift, "2D shifted for FFT (quadrant No.)\n[0,0] at index (1,1)/(0,0)")]):
    ax = fig.add_subplot(gs[1, k])
    ax.imshow(A, cmap="Pastel1", origin="upper")
    for i in range(N2):
        for j in range(N2):
            ax.text(j, i, str(A[i, j]), ha="center", va="center", fontsize=9)
    ax.set_title(ttl); ax.tick_params(labelbottom=False, labelleft=False)

ax = fig.add_subplot(gs[1, 2])
ax.plot(np.abs(G), "s-", ms=5, color="darkorange", mfc="none")
ax.axvline(N2 // 2 - 0.5, color="gray", ls=":", lw=1)
ax.text(N2 // 2 - 0.2, 4.6, "raw FFT output:\nleft half = zero & positive freq,\nright half = negative freq", fontsize=8)
ax.set_title("why fftshift exists: raw fft\nordering vs. centered axis")
ax.set_xticks(np.arange(N2)); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(os.path.dirname(os.path.abspath(__file__)),
                         "out_arrangements.png"), dpi=150)
print("已保存 out_arrangements.png")


## 周期卷积 · 卷积

`ch2_periodic_conv.py`

In [ ]:
# ===== 书 2.6 节：周期卷积（式 2.27，复现图 2.7） =====
# 书图 2.7 参数：L = 20, Δx = 1, N = 20 个样本；
#   f = rect，支撑 D1 = 9（样本 x = -4..4 为 1）；h = 右三角形（上升斜坡），
#   支撑 D2 = 15（x ∈ [-7.5, 7.5]，h(x) = (x+7.5)/15）。
# D1 + D2 = 24 > L = 20 → 违反式 (2.27)：FFT 卷积 ≠ 解析卷积（边缘卷绕污染）。
# 演示两条修复路线：(a) 换更大的 L = 40 重新采样；(b) 同一组样本零填充后再 FFT。
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "common"))
from base import rect, coords


def ramp(x, D):
    """支撑宽度 D 的右三角形（上升斜坡，峰值 1 在右边缘）。"""
    return np.where(np.abs(x) <= D / 2, (x + D / 2) / D, 0.0)


def fft_conv(g, h, dx):
    """FFT 卷积（循环卷积，不填充）：输入居中排列，输出居中排列。"""
    G = np.fft.fft(np.fft.ifftshift(g)) * np.fft.fft(np.fft.ifftshift(h)) * dx
    return np.fft.fftshift(np.fft.ifft(G))


# ---------- 解析参照：细网格直接卷积，再对齐到 xa 坐标 ----------
xa = np.arange(-30.0, 30.0 + 1e-9, 0.005)       # 中心 index 恰为 len//2 ↔ x=0
dd = xa[1] - xa[0]
conv_full = np.convolve(rect(xa / 9.0), ramp(xa, 15.0)) * dd   # 支撑 [-24, 24]
c0 = len(conv_full) // 2                         # conv_full[c0] ↔ t = 0
conv_ref = conv_full[c0 - len(xa) // 2: c0 + len(xa) // 2 + 1]  # 对齐到 xa

# ---------- L = 20：违反式 (2.27) ----------
L, N = 20.0, 20
x, _ = coords(L, N)
g = rect(x / 9.0)            # D1 = 9
h = ramp(x, 15.0)            # D2 = 15
conv_fft = fft_conv(g, h, dx=1.0).real

err20 = np.max(np.abs(np.interp(x, xa, conv_ref) - conv_fft))
pk = np.argmax(conv_ref)
print(f"L=20（D1+D2=24 > L=20，违反式 2.27）：FFT 卷积与解析最大差异 = {err20:.3f}")
print(f"  解析峰值 = {conv_ref[pk]:.3f} @ x = {xa[pk]:.2f}；FFT 峰值 = {conv_fft.max():.3f}")
print("  （书图 2.7(d)：FFT 样本在高频端/边缘跟着周期卷积走，卷绕区多出一块错误面积）")

# ---------- 修复 1：L = 40 重新采样（D1+D2=24 < 40，满足式 2.27） ----------
L2, N2 = 40.0, 40
x2, _ = coords(L2, N2)
conv_fft2 = fft_conv(rect(x2 / 9.0), ramp(x2, 15.0), dx=1.0).real
err40 = np.max(np.abs(np.interp(x2, xa, conv_ref) - conv_fft2))
print(f"L=40（满足式 2.27）：FFT 卷积与解析最大差异 = {err40:.3e}（矩形求和近似误差量级）")

# ---------- 修复 2：零填充（同一组 N=20 样本，填到 N1+N2-1 再 FFT） ----------
P = N + N - 1                                 # 20+20-1 = 39（奇数：中心样本唯一）
g_pad = np.zeros(P); h_pad = np.zeros(P)
o = (P - N) // 2
g_pad[o:o + N] = g; h_pad[o:o + N] = h
Gp = np.fft.fft(np.fft.ifftshift(g_pad)) * np.fft.fft(np.fft.ifftshift(h_pad)) * 1.0
conv_pad = np.fft.fftshift(np.fft.ifft(Gp)).real
xp = (np.arange(P) - P // 2) * 1.0            # 填充网格的居中坐标
err_pad = np.max(np.abs(np.interp(xp, xa, conv_ref) - conv_pad))
print(f"零填充 N1+N2-1={P}：FFT 卷积与解析最大差异 = {err_pad:.3e}")

# ---------- 画图：复现图 2.7(d) + 两种修复 ----------
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0))
axes[0].plot(xa, conv_ref, "k-", lw=1.4, label="analytic")
axes[0].plot(x, conv_fft, "o", ms=6, color="crimson", mfc="none", label="FFT (L=20)")
axes[0].set_title(f"L = 20: D1+D2 = 24 > L\nperiodic wrap-around, max err {err20:.2f}")
axes[1].plot(xa, conv_ref, "k-", lw=1.4, label="analytic")
axes[1].plot(x2, conv_fft2, "o", ms=5, color="teal", mfc="none", label="FFT (L=40)")
axes[1].set_title(f"L = 40: D1+D2 < L satisfied\nmax err {err40:.1e}")
axes[2].plot(xa, conv_ref, "k-", lw=1.4, label="analytic")
axes[2].plot(xp, conv_pad, "o", ms=5, color="darkorange", mfc="none",
             label="zero-padded FFT")
axes[2].set_title(f"zero-padded to N1+N2-1 = {P}\nmax err {err_pad:.1e}")
for ax in axes:
    ax.set_xlim(-30, 30); ax.set_ylim(-0.3, 7); ax.grid(True, alpha=0.3)
    ax.set_xlabel("x"); ax.set_ylabel("(f*h)(x)"); ax.legend(fontsize=8, loc="upper left")
fig.suptitle("Periodic convolution (book Fig. 2.7): rect D1=9 * right-triangle D2=15")
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(os.path.join(os.path.dirname(os.path.abspath(__file__)),
                         "out_periodic_conv.png"), dpi=150)
print("已保存 out_periodic_conv.png")
